# 🦀 Day 2: ORMs — Repository Pattern and Simple Abstraction

---

## Rust ORM Landscape

| ORM | Style | Notes |
|-----|-------|-------|
| **Diesel** | Compile-time query builder | Sync, schema-first, macros |
| **SeaORM** | Async, entity-based | Works with SQLx, async/await |
| **SQLx** | Raw SQL + compile-time checking | Not an ORM, but type-safe queries |

Diesel and SeaORM require code generation and build steps, which don't work well in EvCxR notebooks. So today we'll **build a simple ORM-like abstraction over rusqlite** to learn the concepts!

In [ ]:
:dep rusqlite = { version = "0.31", features = ["bundled"] }

use rusqlite::Connection;

## Repository Pattern

The **Repository pattern** abstracts data access behind a trait. Instead of scattering SQL everywhere, you define operations like `find_all`, `find_by_id`, `create`, `update`, `delete`.

In [ ]:
use rusqlite::{Connection, Result, Row};

// Generic Repository trait — the core ORM concept
trait Repository<T> {
    fn find_all(&self) -> Result<Vec<T>>;
    fn find_by_id(&self, id: i64) -> Result<Option<T>>;
    fn create(&self, entity: &T) -> Result<i64>;
    fn update(&self, entity: &T) -> Result<usize>;
    fn delete(&self, id: i64) -> Result<usize>;
}

## Implementing Repository for a User Entity

We'll implement the trait for a `User` struct backed by SQLite.

In [ ]:
use rusqlite::{Connection, Result, Row};

#[derive(Debug, Clone)]
struct User {
    id: Option<i64>,
    name: String,
    email: String,
}

impl User {
    fn new(name: &str, email: &str) -> Self {
        User { id: None, name: name.to_string(), email: email.to_string() }
    }
    fn from_row(row: &Row) -> Result<Self> {
        Ok(User {
            id: Some(row.get(0)?),
            name: row.get(1)?,
            email: row.get(2)?,
        })
    }
}

struct UserRepository<'a> {
    conn: &'a Connection,
}

impl UserRepository<'_> {
    fn new(conn: &Connection) -> UserRepository {
        UserRepository { conn }
    }
    fn init_schema(&self) -> Result<()> {
        self.conn.execute(
            "CREATE TABLE IF NOT EXISTS users (id INTEGER PRIMARY KEY, name TEXT NOT NULL, email TEXT NOT NULL)",
            [],
        )?;
        Ok(())
    }
}

impl Repository<User> for UserRepository<'_> {
    fn find_all(&self) -> Result<Vec<User>> {
        let mut stmt = self.conn.prepare("SELECT id, name, email FROM users")?;
        let users = stmt.query_map([], User::from_row)?.collect::<Result<Vec<_>>>()?;
        Ok(users)
    }

    fn find_by_id(&self, id: i64) -> Result<Option<User>> {
        let mut stmt = self.conn.prepare("SELECT id, name, email FROM users WHERE id = ?1")?;
        let mut rows = stmt.query(rusqlite::params![id])?;
        if let Some(row) = rows.next()? {
            Ok(Some(User::from_row(&row)?))
        } else {
            Ok(None)
        }
    }

    fn create(&self, entity: &User) -> Result<i64> {
        self.conn.execute(
            "INSERT INTO users (name, email) VALUES (?1, ?2)",
            rusqlite::params![entity.name, entity.email],
        )?;
        Ok(self.conn.last_insert_rowid())
    }

    fn update(&self, entity: &User) -> Result<usize> {
        let id = entity.id.ok_or(rusqlite::Error::InvalidParameterName("id required".into()))?;
        Ok(self.conn.execute(
            "UPDATE users SET name = ?1, email = ?2 WHERE id = ?3",
            rusqlite::params![entity.name, entity.email, id],
        )?)
    }

    fn delete(&self, id: i64) -> Result<usize> {
        Ok(self.conn.execute("DELETE FROM users WHERE id = ?1", rusqlite::params![id])?)
    }
}

let conn = Connection::open_in_memory().unwrap();
let repo = UserRepository::new(&conn);
repo.init_schema().unwrap();

let user = User::new("Alice", "alice@example.com");
let id = repo.create(&user).unwrap();
println!("Created user with id: {}", id);

let found = repo.find_by_id(id).unwrap().unwrap();
println!("Found: {:?}", found);

let all = repo.find_all().unwrap();
println!("All users: {:?}", all);

## Generic Query Builders (Simple Version)

Real ORMs use query builders to compose SQL. Here's a minimal example of the *idea* — a simple filter helper.

In [ ]:
use rusqlite::Connection;

// Simple "query builder" — just a helper to build WHERE clauses
fn find_users_by_email(conn: &Connection, email: &str) -> rusqlite::Result<Vec<(i64, String, String)>> {
    let sql = "SELECT id, name, email FROM users WHERE email = ?1";
    let mut stmt = conn.prepare(sql)?;
    let rows = stmt.query_map(rusqlite::params![email], |row| Ok((row.get(0)?, row.get(1)?, row.get(2)?)))?;
    rows.collect()
}

let conn = Connection::open_in_memory().unwrap();
conn.execute("CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT, email TEXT)", []).unwrap();
conn.execute("INSERT INTO users (name, email) VALUES (?1, ?2)", rusqlite::params!["Alice", "alice@example.com"]).unwrap();
conn.execute("INSERT INTO users (name, email) VALUES (?1, ?2)", rusqlite::params!["Bob", "bob@example.com"]).unwrap();

let alices = find_users_by_email(&conn, "alice@example.com").unwrap();
println!("Users with alice@example.com: {:?}", alices);

## What Diesel / SeaORM Look Like (Reference)

Here's what production ORM code typically looks like. **This won't run in EvCxR** — it's for reference.

**Diesel (sync):**
```rust
// diesel::schema::users::table
users.filter(email.eq("alice@example.com")).load::<User>(&mut conn)?;
diesel::insert_into(users).values(&new_user).execute(&mut conn)?;
```

**SeaORM (async):**
```rust
// Entity-based, async
User::find().filter(user::Column::Email.eq("alice@example.com")).all(db).await?;
User::insert(user_active_model).exec(db).await?;
```

Both use code generation from your schema. Our hand-rolled Repository gives you the same *conceptual* structure.

## 🏋️ Exercise

Implement a `Repository<Product>` for a `Product` entity with `id`, `name`, and `price` (f64). Create the table, implement all CRUD methods, and test them.

In [ ]:
:dep rusqlite = { version = "0.31", features = ["bundled"] }

use rusqlite::{Connection, Result, Row};

trait Repository<T> {
    fn find_all(&self) -> Result<Vec<T>>;
    fn find_by_id(&self, id: i64) -> Result<Option<T>>;
    fn create(&self, entity: &T) -> Result<i64>;
    fn update(&self, entity: &T) -> Result<usize>;
    fn delete(&self, id: i64) -> Result<usize>;
}

#[derive(Debug, Clone)]
struct Product {
    id: Option<i64>,
    name: String,
    price: f64,
}

// YOUR CODE HERE
// 1. Implement Product::new and Product::from_row
// 2. Create ProductRepository with init_schema (CREATE TABLE products)
// 3. Implement Repository<Product> for ProductRepository
// 4. Test: create 2 products, find_all, update one, delete one

## 🎯 Key Takeaways

1. **Rust ORM landscape** — Diesel (sync), SeaORM (async), SQLx (raw SQL)
2. **Repository pattern** — trait-based data access: `find_all`, `find_by_id`, `create`, `update`, `delete`
3. **Implementing Repository** — wrap `Connection`, map rows to structs
4. **Generic query builders** — compose SQL programmatically (we built a simple filter)
5. Diesel/SeaORM use codegen; our hand-rolled version teaches the concepts

---

**Tomorrow:** Migrations and connection pooling 📦